## 나라장터 공고 데이터 보정 프롬프트 가이드

본 노트북은 35건의 공고 데이터를 수동으로 보정하며 얻은 노하우를 바탕으로 작성되었습니다. 향후 AI 에이전트 자동화의 기초 설계도로 사용됩니다.

### 1. 검색 에이전트 프롬프트 (Search Agent)

- **목적:** 나라장터 상세 검색을 통해 정확한 공고 페이지에 접속하기 위한 지침
- **핵심 전략:** 공고기관명과 사업명을 조합하여 검색 정확도 향상

Role: 나라장터 입찰 데이터 수집 전문가 (G2B Data Scraper)

Context

당신은 국가종합전자조달시스템(나라장터)에서 특정 입찰 공고의 누락된 메타데이터를 찾아 보충하는 업무를 수행합니다.

Search Protocol (표준 검색 경로)

1. 접속: 나라장터 홈페이지 (g2b.go.kr)

2. 이동: 입찰정보 -> 용역 -> 공고현황 -> 상세검색

3. 입력:

    * [공고명]: 제공된 '나라장터 검색어' 입력

    * [공고연도]: 제공된 '검색 범위'의 연도 설정

4. 실행: 검색 버튼 클릭 후 해당 공고의 '공고번호' 클릭하여 상세페이지 진입

- Task: 데이터 추출 및 보정 (상세페이지에서 다음 항목을 찾아 정해진 형식으로 추출하십시오.)

    - 입찰참여 시작일: [YYYY-MM-DD HH:MM] 형식 (예: 2025-05-10 10:00)

    - 입찰참여 마감일: [YYYY-MM-DD HH:MM] 형식

    - 사업 금액(추정가격/예정가격): 숫자만 추출 (예: 150,000,000)

- Example Data (Few-shot)

    - 검색 대상: 강릉어선안전조업국 상황관제시스템 구축

    - 검색어: 수협중앙회 강릉어선안전조업국 상황관제시스템 구축

    - 보정 항목: 입찰 참여 시작일

- 출력 요구: 해당 공고의 '입찰집행/진행내역' 탭 또는 본문에서 '입찰서접수개시일시'를 찾아 기록할 것.

## 2. 데이터 추출 프롬프트 (Data Extraction)

- **목적:** 공고문 텍스트에서 입찰일, 예산 등 핵심 메타데이터 추출
- **핵심 전략:** 날짜 형식(YYYY-MM-DD) 통일 및 숫자 데이터 정제

Prompt:  
"아래는 나라장터에서 긁어온 공고문 텍스트야. 여기서 '입찰서 접수 시작일'과 '마감일', 그리고 '배정예산'을 찾아서 JSON 형식으로 출력해줘. 만약 금액에 콤마가 있으면 제거하고 숫자만 남겨줘. 날짜는 하이픈(-)으로 구분된 표준 형식이어야 해."

## 3. 수동 보정 시 발견된 예외 케이스 (Insights)

사례 1: 본문에 금액이 없을 경우 첨부파일 확인 필요

사례 2: 공고명만으로 검색되지 않을 시 공고기관 필터 추가 설정

### 3. 수동 보정 시 발견된 예외 케이스 및 처리 로직

수동 보정 과정에서 웹 UI의 표준 필드에 데이터가 누락되거나 왜곡된 경우 아래의 예외 처리 규칙에 따라 데이터를 보정하였습니다. 본 가이드는 향후 AI 에이전트의 데이터 추출 로직 설계 및 데이터 클리닝의 최우선 지침으로 활용됩니다.

| 예외 유형 | 발생 현상 및 원인 | 보정 규칙 (Logic) |
| --- | --- | --- |
| **공개일자 데이터 불일치** | 공개일자와 실제 입찰 참여 시작일의 격차가 크거나 시스템 필드 값이 부정확함 | **처리**: 공개일자는 개별 수집 항목에서 **제외**함. 대신 데이터 수집 시 **검색 범위(1년 단위)** 를 설정하기 위한 필터링 조건으로만 활용함 |
| **데이터 신뢰성 불일치** | 발주기관 메타데이터와 파일 본문 내용이 상이한 경우 | **처리**: 파일 본문과 발주기관을 교차 검증하여 일치하지 않는 파일은 제거(Drop)함 |
| **사업비 특이 케이스 (0/1)** | 시스템상 예산이 누락되었거나 비공개인 경우 | **0 (미기입)**: 상세 페이지 내 **용역비용** 등에서 찾아 실금액으로 수정
|||**1 (비공개)**: 공고문 내에서도 확인 불가능한 비공개 건은 **1**로 기입하여 관리 |
| **사업비 우선순위** | 배정예산과 추정가격이 동시에 존재하는 경우 | **배정예산(VAT 포함)** 을 최우선으로 기록하여 실제 총 사업 규모 반영 |
| **추정가격만 존재** | 배정예산 필드가 비어 있고 추정가격만 명시된 경우 | **추정가격**을 기록하되 비고란에 **VAT 제외 금액**임을 명시 |
| **입찰한도액 표기** | 사업 금액 항목 대신 입찰한도액으로 명시된 경우 | **Budget**: 입찰한도액을 해당 사업의 예산(숫자)으로 기록 |
| **정정공고** | 내용 오류 수정이나 일정 변경으로 수정 공고가 올라온 경우 | **번호**: 최신 정정 번호 적용
|||**시작일**: **최초 공고 게시일** 적용
|||**종료일**: **정정된 최종 마감일** 적용 |
| **재공고 사업** | 유찰 등으로 동일 사업이 다시 공고된 경우 | **번호**: 최신 재공고 번호 적용
|||**시작일**: **최초 공고 게시일** 적용
|||**종료일**: **최종 재공고의 마감일** 적용 |
| **직찰/집찰 공고** | 입찰 방법 특성상 시스템 날짜 필드가 누락된 경우 | **시작일**: 상세 일정 테이블의 **공고 게시** 일시 적용
|||**종료일**: 상세 일정 테이블의 **개찰** 또는 **마감** 일시 적용 |
| **공고번호 왜곡** | 11자리 이상의 번호가 엑셀에서 지수(E+) 형태로 변형됨 | **처리**: 셀 서식을 **텍스트(Text)** 로 강제 지정하여 데이터 유실 및 뒷자리 변조 방지 |
| **공고서 참조 일정** | 시스템 필드 대신 첨부파일 내부에만 일정이 명시된 경우 | **공고서(PDF/HWP)** 본문 내 일정 테이블을 참조하여 **제안서 마감일**을 종료일로 설정 |
| **검색어 불일치** | 사업명만으로 검색 시 유사 공고가 너무 많이 노출되는 경우 | **공고기관명 + 사업명** 조합으로 검색어를 재구성하여 정확도 향상 |

**비고 및 인사이트**

- **날짜 기준 정립**: 공개일자는 검색 범위를 1년 단위로 한정하는 용도로만 사용하며 실제 분석용 시작일은 상세 일정의 **공고 게시** 일자를 기준으로 함
- **신뢰성 검증**: 발주기관과 파일 본문이 다른 문서는 신뢰도 문제로 데이터셋에서 즉시 제외함
- **사업비 구분**: **0**은 보정 대상 데이터 **1**은 분석 제외 또는 비공개 데이터로 분류하여 통계의 정확성을 확보함

**데이터 무결성 주의 사항**

- **공고번호 보존**: 11자리 이상의 번호는 지수 형태로 저장 시 뒷자리가 유실되므로 반드시 엑셀 셀 왼쪽 상단의 **초록색 삼각형**이 보이는 **텍스트 상태**로 관리함
- **파이프라인 연동**: CSV 로드 시 공고번호 컬럼을 **문자열(str)** 타입으로 지정하여 데이터 왜곡을 방지함
- **통계 예외 처리**: 사업비가 **0** 또는 **1**로 기록된 데이터는 평균이나 합계 계산 시 별도의 필터링 처리가 필요함

> ### 데이터 제외 리스트
| 발주 기관 | 사업명 | 파일명 | 사유 |
| --- | --- | --- | --- |
| 국가과학기술지식정보서비스 | 통합정보시스템 고도화 용역 | 국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp | 한국한의학연구원의 발주 파일과 같은 내용이며, 나라장터 시스템에서도 한국한의학연구원의 발주로만 탐색됨 |
